In [1]:
import ollama 
import os
from tqdm import tqdm
import json
import signal
import argparse
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import copy

import sys
from collections import defaultdict

#from  utilities import *

In [2]:
from PIL import Image
import matplotlib.pyplot as plt

In [3]:
list_of_models = ['llava:7b', 'llava:13b', 'llava:34b', 'llama3.2-vision:90b']

In [27]:
sys.argv = [
    'notebook',  
    '--modelname', 'llava:34b',
    '--data', 'gully',
    '--data_path','/home/data/',
    "--data_samples", 'train.jsonl',
    "--subset", 'train',
    '--results_dir', '/mnt/jacket/WACV-2025-Workshop-ViGIR/results/baseline',
    '--timeout', '20',
    '--model_unloading'
]

In [28]:
parser = argparse.ArgumentParser(description="A script to run V-LLMs on different image classification datasets")

In [29]:
# set up the argument parser
parser.add_argument("--modelname", type=str, required=True, help="The name of the V-LLM model")
parser.add_argument("--data", type=str, required=True, help="Dataset name")
parser.add_argument("--data_path", type=str, required=True, help="Path to the image data dir")
parser.add_argument("--data_samples", type=str, required=True, help="Name of the samples to run on")
parser.add_argument("--subset", type=str, required=True, help="train, test or validation set")
parser.add_argument("--results_dir", type=str, required=True, help="Folder name to save results")
parser.add_argument("--timeout", type=int, default=40, help="time out duration to skip one sample")
parser.add_argument("--model_unloading", action="store_true", help="Enables unloading mode. Every 100 sampels it unloades the model from the GPU to avoid carshing.")


args = parser.parse_args()

In [30]:
# Create a result folder
print(args.results_dir)
if not os.path.exists(args.results_dir):
    os.makedirs(args.results_dir)

/mnt/jacket/WACV-2025-Workshop-ViGIR/results/baseline


In [31]:
# Load test set:
file_path = 'Testing_Set.json'
#file_path = 'Validation_Set.json'
with open(file_path, 'r') as file:
    data = json.load(file)

print('Number of Annotated GT Images: ', len(data.keys()))
data_keys_test = list(data.keys())
print('Number of Annotated GT Images (List): ', len(data_keys_test))

Number of Annotated GT Images:  311
Number of Annotated GT Images (List):  311


In [32]:
# Load the images
directory = '/root/home/data_jpg'
tile_dict = defaultdict(list)

for filename in os.listdir(directory):
    if filename.endswith('.jpg'):
        parts = filename.split('_')
        tile_number = parts[4].split('.')[0]
        tile_dict[tile_number].append(os.path.join(directory,filename))

tile_dict = dict(tile_dict)

print("Total number of tiles: {}".format(len(tile_dict.keys())))

Total number of tiles: 900


In [33]:
tile_dict

{'100': ['/root/home/data_jpg/neg_rgb_0_tile_100.jpg',
  '/root/home/data_jpg/neg_rgb_1_tile_100.jpg',
  '/root/home/data_jpg/neg_rgb_2_tile_100.jpg',
  '/root/home/data_jpg/neg_rgb_4_tile_100.jpg',
  '/root/home/data_jpg/neg_rgb_3_tile_100.jpg',
  '/root/home/data_jpg/neg_rgb_5_tile_100.jpg'],
 '1000': ['/root/home/data_jpg/neg_rgb_0_tile_1000.jpg',
  '/root/home/data_jpg/neg_rgb_1_tile_1000.jpg',
  '/root/home/data_jpg/neg_rgb_2_tile_1000.jpg',
  '/root/home/data_jpg/neg_rgb_4_tile_1000.jpg',
  '/root/home/data_jpg/neg_rgb_3_tile_1000.jpg',
  '/root/home/data_jpg/neg_rgb_5_tile_1000.jpg'],
 '1001': ['/root/home/data_jpg/neg_rgb_0_tile_1001.jpg',
  '/root/home/data_jpg/neg_rgb_1_tile_1001.jpg',
  '/root/home/data_jpg/neg_rgb_2_tile_1001.jpg',
  '/root/home/data_jpg/neg_rgb_4_tile_1001.jpg',
  '/root/home/data_jpg/neg_rgb_3_tile_1001.jpg',
  '/root/home/data_jpg/neg_rgb_5_tile_1001.jpg'],
 '1002': ['/root/home/data_jpg/neg_rgb_0_tile_1002.jpg',
  '/root/home/data_jpg/neg_rgb_1_tile_100

In [34]:
for tile_number, file_list in tile_dict.items():
    tile_dict[tile_number] = sorted(file_list, key=lambda x: int(x.split('_')[3]))

In [35]:
tile_dict_val = {}
for tile_number, file_list in tile_dict.items():
    if tile_number in data_keys_test:
        #tile_dict[tile_number] = sorted(file_list, key=lambda x: int(x.split('_')[2]))
        tile_dict_val[tile_number] = file_list

In [36]:
print('Number of Retained Annotated GT Images: ', len(tile_dict_val.keys()))

Number of Retained Annotated GT Images:  311


In [37]:
data = copy.deepcopy(tile_dict_val)

In [41]:
questions = ["Given the six images of the exact same area, collected over a period of 10 years. Are there any ephemeral gully appearances by looking at all of them together?. Reason and conclude with only yes or no. "]


In [42]:
model_name = args.modelname
ollama.pull(model_name)

timeout_duration = args.timeout

options= {  # new
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048, # must be set, otherwise slightly random output
        }

model_labels = {}
count = 0

In [43]:
count = 0
saving_response=dict(copy.deepcopy(tile_dict_val))

for key, info in tqdm(data.items()):

    #print(key)
    #print(info)
    #sys.exit()
    
    count+=1
    for question in questions:
        response = ollama.generate(model=model_name, 
                                   prompt=question, 
                                   images=info, 
                                   options=options)
        saving_response[key].append(response['response'])
        print(response['response'])

        #sys.exit()
        
       
    

  0%|▎                                                                                                     | 1/311 [00:04<24:15,  4.70s/it]

Based on the provided images, it appears that there are some changes in the landscape over the 10-year period. Specifically, there seems to be an ephemeral gully formation visible in the middle of the image series. This conclusion is based on the observation that a distinct channel or depression has appeared between the years, indicating erosion and possibly rainfall events that have caused soil movement.

Therefore, I conclude with "Yes" as there are apparent changes in the landscape over time, including the appearance of an ephemeral gully.


  1%|▋                                                                                                     | 2/311 [00:09<25:12,  4.89s/it]

Based on the provided images, it appears that there are some changes in the landscape over the 10-year period. Specifically, there seems to be an increase in gully erosion, which is a form of ephemeral land degradation. The appearance and extent of these gullies have likely been influenced by factors such as rainfall patterns, soil characteristics, vegetation cover, and human activities that affect the stability of the slopes.

Therefore, I conclude with "Yes" to indicate that there are visible changes in the landscape, including the ephemeral appearances of gully erosion when looking at all six images together over time.


  1%|▉                                                                                                     | 3/311 [00:15<26:21,  5.13s/it]

...

Based on the provided images, it appears that there are some changes in the area over the 10-year period. Specifically, there seems to be an ephemeral gully appearance in the middle of the series (around image 3 or 4) where a small incision or depression has formed compared to the surrounding areas. This suggests that there might have been a temporary erosion event or some localized disturbance that led to this change.

However, without more detailed analysis and additional context such as climate data, land use changes, or geological activity, it's difficult to provide a definitive conclusion on whether these changes are truly ephemeral or indicative of a longer-term process. Therefore, I can only conclude with "Yes" in the sense that there is some evidence of change within the series of images provided.


100%|███████████████████████████████████████████████████████████████████████████████████████████████████▋| 310/311 [26:23<00:05,  5.11s/it]

Based on the provided images, it appears that there is an ephemeral gully formation visible in some of the images but not consistently present across all six images. The gully seems to be more pronounced in certain years and less so in others. This suggests that the gully may have been a temporary feature or subject to changes over time due to factors such as rainfall patterns, soil erosion rates, and vegetation cover.

Therefore, I conclude with "Yes" because there is evidence of ephemeral gully appearances when looking at all six images together.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 311/311 [26:28<00:00,  5.11s/it]

Based on the provided images, it appears that there are some changes in the landscape over the 10-year period. Specifically, there seems to be a development of gullies or erosion features in the area. These gullies appear to have formed gradually and may not be immediately noticeable when looking at individual images but become more apparent when comparing them side by side.

Therefore, I conclude that yes, there are ephemeral gully appearances evident when considering all six images together over the 10-year period. This suggests a process of erosion or land degradation in this particular area.


In [44]:
saving_response

{'100': ['/root/home/data_jpg/neg_rgb_0_tile_100.jpg',
  '/root/home/data_jpg/neg_rgb_1_tile_100.jpg',
  '/root/home/data_jpg/neg_rgb_2_tile_100.jpg',
  '/root/home/data_jpg/neg_rgb_3_tile_100.jpg',
  '/root/home/data_jpg/neg_rgb_4_tile_100.jpg',
  '/root/home/data_jpg/neg_rgb_5_tile_100.jpg',
  'Based on the provided images, it appears that there are some changes in the landscape over the 10-year period. Specifically, there seems to be an ephemeral gully formation visible in the middle of the image series. This conclusion is based on the observation that a distinct channel or depression has appeared between the years, indicating erosion and possibly rainfall events that have caused soil movement.\n\nTherefore, I conclude with "Yes" as there are apparent changes in the landscape over time, including the appearance of an ephemeral gully.'],
 '1004': ['/root/home/data_jpg/neg_rgb_0_tile_1004.jpg',
  '/root/home/data_jpg/neg_rgb_1_tile_1004.jpg',
  '/root/home/data_jpg/neg_rgb_2_tile_1004

In [46]:
# Save the dictionary as a JSON file
filename = 'reason_six_results_'+args.modelname+'.json'
with open(os.path.join(args.results_dir, filename), 'w') as json_file:
    json.dump(saving_response, json_file, indent=4)

print(f"JSON file {filename} has been created.")

JSON file reason_six_results_llava:34b.json has been created.
